# 0.Library

In [1]:
import pandas as pd
from pathlib import Path

import importlib
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp

# Reload
importlib.reload(du)
importlib.reload(dp)

# Constants and paths
parent_folder = Path("../..") # go 2 folder up: "../.."
data_folder = parent_folder / "data"
input_path_of_json = data_folder / "patients_data_log.json"
df_data_log = pd.read_json(input_path_of_json)

patient_ids = df_data_log["PatientID"].to_list()
patients_data_folder = data_folder / "PatientsData"

# 1.

In [2]:
# get clean path of one patient (all csv files)
patient_id = patient_ids[0]
patient_folder_path = du.find_patient_folder(patients_data_folder=patients_data_folder,
                                               patient_id=patient_id)
                                               
path_to_check = patient_folder_path / "clean_data"
clean_csv_files = du.find_csv_file(patient_folder_path / "clean_data")
clean_csv_files

['cleaned_20260502_113117_01_Tutorial_events.csv',
 'cleaned_20260502_113117_01_Tutorial_trajectory.csv',
 'cleaned_20260502_113117_02_ObjectRecognition_events.csv',
 'cleaned_20260502_113117_02_ObjectRecognition_trajectory.csv',
 'cleaned_20260502_113117_03_Visuospatial_events.csv',
 'cleaned_20260502_113117_03_Visuospatial_trajectory.csv',
 'cleaned_20260502_113117_04_Memory_events.csv',
 'cleaned_20260502_113117_04_Memory_trajectory.csv']

In [3]:
# start with the first csv file
clean_df = pd.read_csv(path_to_check / clean_csv_files[0])

In [4]:
clean_df.head()

,Timestamp,PhaseTime_s,Section,EventType,ObjectName,Hand,IsCorrect,Duration_s,Distance_m,Details,Activity_Log
0,2026-05-02 11:31:17.628,0.000,Default,PHASE_START,NaN,NaN,NaN,0.0,0.0,Phase=01_Tutorial,"['Section=Default', 'SessionID=20260502_113117..."
1,2026-05-02 11:31:17.630,0.000,Default,PANEL_SHOWN,WelcomePanel,NaN,NaN,0.0,0.0,Section=Default,['Panel=WelcomePanel']
2,2026-05-02 11:31:22.971,0.000,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
3,2026-05-02 11:31:23.918,0.952,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
4,2026-05-02 11:31:26.072,3.106,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...


In [5]:
clean_df.columns

Index(['Timestamp', 'PhaseTime_s', 'Section', 'EventType', 'ObjectName',
       'Hand', 'IsCorrect', 'Duration_s', 'Distance_m', 'Details',
       'Activity_Log'],
      dtype='object')

In [ ]:
# work on one section
section_names = clean_df['Section'].unique()
section_names

array(['Default', 'ButtonSelection', 'GrabPractice', 'Completion'],
      dtype=object)

In [22]:
section_df = du.extract_section(clean_df, section_names[0])
section_df.head()

,Timestamp,PhaseTime_s,Section,EventType,ObjectName,Hand,IsCorrect,Duration_s,Distance_m,Details,Activity_Log
0,2026-05-02 11:31:17.628,0.000,Default,PHASE_START,NaN,NaN,NaN,0.0,0.0,Phase=01_Tutorial,"['Section=Default', 'SessionID=20260502_113117..."
1,2026-05-02 11:31:17.630,0.000,Default,PANEL_SHOWN,WelcomePanel,NaN,NaN,0.0,0.0,Section=Default,['Panel=WelcomePanel']
2,2026-05-02 11:31:22.971,0.000,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
3,2026-05-02 11:31:23.918,0.952,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
4,2026-05-02 11:31:26.072,3.106,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...


In [ ]:
# extract event types (cleanning base on event types)
events_to_ignore = ['PHASE_START','SECTION_END' ]
event_types = du.extract_event_types(section_df, events_to_ignore)
event_types

['PANEL_SHOWN',
 'BUTTON_HOVER_START',
 'BUTTON_HOVER_END',
 'BUTTON_PRESSED',
 'BUTTON_RELEASED',
 'PANEL_DISMISSED']

In [23]:
# create dict for each event type
dict_hover_button = {
    'button_name' : None,
    'hover_count' : None,
    'hover_duration' : None
}

dict_press_button = {
    'button_name' : None,
    'press_duration' : None
}

dict_panel = {
    'panel_name' : None,
    'reading_time' : None
}

In [24]:
clean_df.head(30)

,Timestamp,PhaseTime_s,Section,EventType,ObjectName,Hand,IsCorrect,Duration_s,Distance_m,Details,Activity_Log
0,2026-05-02 11:31:17.628,0.000,Default,PHASE_START,NaN,NaN,NaN,0.000,0.0,Phase=01_Tutorial,"['Section=Default', 'SessionID=20260502_113117..."
1,2026-05-02 11:31:17.630,0.000,Default,PANEL_SHOWN,WelcomePanel,NaN,NaN,0.000,0.0,Section=Default,['Panel=WelcomePanel']
2,2026-05-02 11:31:22.971,0.000,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
3,2026-05-02 11:31:23.918,0.952,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
4,2026-05-02 11:31:26.072,3.106,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
5,2026-05-02 11:31:26.183,3.217,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
6,2026-05-02 11:31:26.378,3.411,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
7,2026-05-02 11:31:26.697,3.731,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
8,2026-05-02 11:31:28.907,5.940,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
9,2026-05-02 11:31:29.448,6.483,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...


In [25]:
section_df

,Timestamp,PhaseTime_s,Section,EventType,ObjectName,Hand,IsCorrect,Duration_s,Distance_m,Details,Activity_Log
0,2026-05-02 11:31:17.628,0.000,Default,PHASE_START,NaN,NaN,NaN,0.000,0.0,Phase=01_Tutorial,"['Section=Default', 'SessionID=20260502_113117..."
1,2026-05-02 11:31:17.630,0.000,Default,PANEL_SHOWN,WelcomePanel,NaN,NaN,0.000,0.0,Section=Default,['Panel=WelcomePanel']
2,2026-05-02 11:31:22.971,0.000,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
3,2026-05-02 11:31:23.918,0.952,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
4,2026-05-02 11:31:26.072,3.106,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
5,2026-05-02 11:31:26.183,3.217,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
6,2026-05-02 11:31:26.378,3.411,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
7,2026-05-02 11:31:26.697,3.731,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
8,2026-05-02 11:31:28.907,5.940,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
9,2026-05-02 11:31:29.448,6.483,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.000,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...


In [32]:
x = section_df[section_df['Activity_Log'].str.contains('HoverCount')]['Activity_Log']

In [33]:
for v in x:
    print(v)

['Button=StartTutorialButton[TutorialWelcome]', 'HoverCount=1']
['Button=StartTutorialButton[TutorialWelcome]', 'HoverCount=2']
['Button=StartTutorialButton[TutorialWelcome]', 'HoverCount=3']
['Button=StartTutorialButton[TutorialWelcome]', 'HoverCount=4']
['Button=StartTutorialButton[TutorialWelcome]', 'HoverCount=5']
['Button=StartTutorialButton[TutorialWelcome]', 'HoverCount=6']
['Button=StartTutorialButton[TutorialWelcome]', 'HoverCount=7']


In [51]:
def extract_hover_count(s):
    import re
    match = re.search(r"HoverCount=(\d+)", s)
    return int(match.group(1)) if match else None

In [52]:
def extract_hover_duration(s):
    import re
    match = re.search(r"HoverDuration=([0-9]*\.?[0-9]+)", s)
    return float(match.group(1)) if match else None

In [ ]:
def extract_press_count(s):
    pass

In [58]:
def extract_press_duration(s):
    import re
    match = re.search(r"PressDuration=([0-9]*\.?[0-9]+)", s)
    return float(match.group(1)) if match else None

In [45]:
def extract_metric_from_section(s, function_to_extract_metric):
    section_metric_values = []
    for value in s:
        print(function_to_extract_metric(value))
        section_metric_values.append(function_to_extract_metric(value))

    section_metric_values_pandas = pd.Series(section_metric_values)
    return section_metric_values_pandas


In [36]:
def filter_by_string_contains(df, column_name, substring):
    return df[df[column_name].str.contains(substring, na=False)]

In [53]:
# hover count
filtered_df = filter_by_string_contains(section_df, 'Activity_Log', 'HoverCount')
hover_count_series = extract_metric_from_section(filtered_df['Activity_Log'], extract_hover_count)

1
2
3
4
5
6
7


In [ ]:
#hover duration
filtered_df = filter_by_string_contains(section_df, 'Activity_Log', 'HoverDuration')
hover_duration_series = extract_metric_from_section(filtered_df['Activity_Log'], extract_hover_duration)

0.95
0.11
0.32
0.54
2.29
0.49
0.82


In [55]:
def calculate_metric_stats(metric_series, is_counting=True):

    # calculate total sum for both counting and duration metrics
    total = metric_series.sum()
    print("Sum:", total)

    # For counting metrics, we only care about the total sum.
    if is_counting:
        total = metric_series.sum()
        return total, None, None, None, None
    
    # For duration metrics, we calculate all statistics.
    mean = metric_series.mean()
    maximum = metric_series.max()
    median = metric_series.median()
    std = metric_series.std()

    print("Mean:", mean)
    print("Max:", maximum)
    print("Median:", median)
    print("Std:", std)

    return total, mean, maximum, median, std

In [56]:
# hover_count
calculate_metric_stats(hover_count_series, is_counting=True)

Sum: 28


(np.int64(28), None, None, None, None)

In [57]:
# calcuation for hover duration
calculate_metric_stats(hover_duration_series, is_counting=False)

Sum: 5.5200000000000005
Mean: 0.7885714285714286
Max: 2.29
Median: 0.54
Std: 0.7202182208985591


(np.float64(5.5200000000000005),
 np.float64(0.7885714285714286),
 np.float64(2.29),
 np.float64(0.54),
 np.float64(0.7202182208985591))

In [59]:
# calculation for press count
pass

In [60]:
#press duration
filtered_df = filter_by_string_contains(section_df, 'Activity_Log', 'PressDuration')
press_duration_series = extract_metric_from_section(filtered_df['Activity_Log'], extract_press_duration)

0.222


In [61]:
calculate_metric_stats(press_duration_series, is_counting=False)

Sum: 0.222
Mean: 0.222
Max: 0.222
Median: 0.222
Std: nan


(np.float64(0.222),
 np.float64(0.222),
 np.float64(0.222),
 np.float64(0.222),
 np.float64(nan))

In [ ]:
# reading time duration
filtered_df = filter_by_string_contains(section_df, 'Activity_Log', 'ReadingTime')
reading_time_series = extract_metric_from_section(filtered_df['Activity_Log'], extract_press_duration)